# Baseline en test con LM Studio

Evalúa los modelos sin entrenar sobre todo `datos/test.jsonl`: 20 árboles en cada uno de los 8 niveles, con `INTENTOS` intentos por árbol. Los episodios ya guardados no se repiten. Al terminar un modelo, sigue con el siguiente de `MODELOS`.

**Antes de empezar:** abre LM Studio, inicia el servidor local y evita que el PC entre en suspensión. LM Studio carga cada modelo cuando le toca. Abre este notebook desde la carpeta `seleccion_arbol` y selecciona el kernel del entorno de conda `tipico`, que tiene `pandas` y `numpy`.

**Si se interrumpe:** ejecuta todo de nuevo. Cada episodio se guarda al terminar, así que continúa donde quedó. No cambies el `nombre` de un modelo con episodios guardados, o empezaría de cero en otra carpeta.

**Intentos 4 a 6:** DeepSeek, Llama y OLMo ya tienen una segunda corrida en los niveles 2 a 6, guardada como intentos 4 a 6. Con `INTENTOS = 4`, en esos niveles no se corre nada nuevo.

**Si cambias un presupuesto:** usa otra carpeta de resultados, para no mezclar corridas.

## Configuración

In [ ]:
from evaluacion import leer_resultados
from modelo import DEEPSEEK, LLAMA, OLMO, QWEN, ModeloLMStudio

URL = 'http://localhost:1235/v1'
RUTA_DATOS = 'datos/test.jsonl'
NIVELES = sorted({ejemplo['nivel'] for ejemplo in leer_resultados(RUTA_DATOS)})  # Todos los niveles del test
INTENTOS = 3                                    # Intentos por árbol. Con 4, solo corre el cuarto donde falta
TOKENS_PENSAMIENTO = 3400                       # Al agotarlos, se cierra el razonamiento y el modelo debe responder
TOKENS_RESPUESTA = 200                          # Tokens para escribir los nodos después de ese cierre

# Se evalúan en este orden. Un formato incorrecto no da error, pero altera los resultados.
# El nombre es la columna `modelo` y la carpeta de resultados: debe coincidir con la que ya existe.
MODELOS = [
    dict(nombre='DeepSeek R1 Distill 1.5B', lm_studio='deepseek-r1-distill-qwen-1.5b', formato=DEEPSEEK),
    dict(nombre='Qwen 3.5 2B', lm_studio='qwen3.5-2b', formato=QWEN),
    dict(nombre='OLMo 2 1B', lm_studio='olmo-2-0425-1b-instruct', formato=OLMO),
    dict(nombre='Llama 3.2 1B', lm_studio='llama-3.2-1b-instruct', formato=LLAMA),
]
RUTAS = [f"resultados/{m['nombre']}/baseline_test.jsonl" for m in MODELOS]

## Comprobar la conexión

Si esta celda falla, revisa que el servidor de LM Studio esté iniciado.

In [ ]:
import json
import urllib.request

with urllib.request.urlopen(f'{URL}/models', timeout=10) as respuesta:
    disponibles = [m['id'] for m in json.load(respuesta)['data']]

print('Modelos disponibles:', disponibles)
faltantes = [m['lm_studio'] for m in MODELOS if m['lm_studio'] not in disponibles]
assert not faltantes, f'No están disponibles en LM Studio: {faltantes}'

## Datos de test

Resumen por nivel y todos los árboles del test, incluidos los niveles 1, 7 y 8. La tabla completa omite `prompt` y `razonamiento`, que son textos largos.

- **atajo_optimo:** árboles donde la estrategia "sin vecinos" ya alcanza el óptimo.

In [ ]:
import pandas as pd

test = pd.DataFrame(leer_resultados(RUTA_DATOS))
test['atajo_optimo'] = test['puntaje_sin_vecinos'] == test['puntaje_optimo']
test.groupby('nivel').agg(
    arboles=('id', 'size'),
    nodos=('nodos', 'first'),
    costo_medio=('costo', 'mean'),
    optimo_medio=('puntaje_optimo', 'mean'),
    voraz_medio=('puntaje_voraz', 'mean'),
    sin_vecinos_medio=('puntaje_sin_vecinos', 'mean'),
    atajo_optimo=('atajo_optimo', 'mean'),
).round(2)

In [ ]:
numero = test['id'].str.rsplit('-', n=1).str[1].astype(int)
tabla_test = test.assign(numero=numero).sort_values(['nivel', 'numero']).drop(columns=['numero', 'prompt', 'razonamiento'])

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(tabla_test.reset_index(drop=True))

## Evaluación

Esta celda sigue hasta terminar todos los modelos. Si un modelo falla, pasa al siguiente y lo reintenta al final, hasta 5 rondas con 5 minutos de pausa.

Para ver los resultados antes de que termine, interrumpe la celda y ejecuta las de abajo. Al volver a ejecutarla, continúa donde quedó y solo se repite el episodio en curso.

In [ ]:
from evaluacion import evaluar_varios

evaluaciones = [
    (
        ModeloLMStudio(
            nombre=m['lm_studio'],
            formato=m['formato'],
            url_base=URL,
            tokens_pensamiento=TOKENS_PENSAMIENTO,
            tokens_respuesta=TOKENS_RESPUESTA,
        ),
        m['nombre'],
        ruta,
    )
    for m, ruta in zip(MODELOS, RUTAS)
]

incompletos = evaluar_varios(evaluaciones, RUTA_DATOS, niveles=NIVELES, intentos=INTENTOS)
print('Modelos con episodios pendientes:', incompletos or 'ninguno')

## Baseline por nivel

Aparecen todos los modelos y niveles: los que aún no se evalúan quedan vacíos. No ejecutes la celda de evaluación en dos kernels a la vez: se repetirían episodios.

Las columnas de proporción van de 0 a 1.

- **pass@1:** intentos que logran el óptimo.
- **pass@3:** árboles donde al menos uno de sus 3 intentos logra el óptimo.
- **recompensa:** media del episodio. Igualar el mejor atajo vale 0,5, y una respuesta inválida, -1.
- **optimos:** árboles donde el modelo logró el mejor puntaje posible.
- **optimos_atajo:** óptimos de la estrategia "sin vecinos", que no razona sobre costos. El modelo debería superarla para mostrar que optimiza de verdad.
- **supera_voraz:** veces que superó la regla simple.
- **supera_atajos:** veces que superó al mejor atajo.
- **invalidos:** respuestas sin un formato válido.
- **cortes:** episodios que agotaron el pensamiento, y cuya respuesta fue forzada.

In [ ]:
import pandas as pd

from evaluacion import resumir, resumir_pass

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
todos = pd.MultiIndex.from_product([[m['nombre'] for m in MODELOS], NIVELES], names=['modelo', 'nivel'])

resumir_pass(RUTAS, k=INTENTOS).reindex(todos)

In [ ]:
resumir(RUTAS).reindex(todos)

## Baseline total

Mientras un modelo no complete todos los niveles, su total no se compara con el de los demás, porque promedia niveles distintos.

In [ ]:
resumir_pass(RUTAS, k=INTENTOS, por=['modelo'])

In [ ]:
resumir(RUTAS, por=['modelo'])